<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

fatal: destination path 'ML_fx' already exists and is not an empty directory.


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
test=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "test_prepared.csv"))
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))

print(len(test))
print(test.head())

115064
   Store  Dept        Date  IsHoliday  Temperature  Fuel_Price  MarkDown1  \
0      1     1  2012-11-02          0        55.32       3.386    6766.44   
1      1     1  2012-11-09          0        61.24       3.314   11421.32   
2      1     1  2012-11-16          0        52.92       3.252    9696.28   
3      1     1  2012-11-23          1        56.23       3.211     883.59   
4      1     1  2012-11-30          0        52.34       3.207    2460.03   

   MarkDown2  MarkDown3  MarkDown4  ...  Type    Size  sales_lag_52  Year  \
0    5147.70      50.82    3639.90  ...    20  151315      39886.06  2012   
1    3370.89      40.28    4646.79  ...    20  151315      18689.54  2012   
2     292.10     103.78    1133.15  ...    20  151315      19050.66  2012   
3       4.17   74910.32     209.91  ...    20  151315      20911.25  2012   
4     192.00    3838.35     150.57  ...    20  151315      25293.49  2012   

   month_sin  month_cos   dow_sin   dow_cos  week_sin  week_cos  
0

In [ ]:


!pip install wandb -q
!pip install neuralforecast wandb -q
!pip install pytorch-lightning==2.0.6

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.8/722.8 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.5.6
    Uninstalling pytorch-lightning-2.5.6:
      Successfully uninstalled pytorch-lightning-2.5.6


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [24]:
import wandb
import joblib
import os

# 1. Initialize a run (required to download artifacts)
run = wandb.init(project="ML_fx_Prophet_Walmart", name="Fetch_LGBM_From_Registry")

# 2. Define the artifact path in the registry
# Format: wandb-registry-{ENTITY}/{PROJECT_NAME}/{COLLECTION_NAME}:{ALIAS}
ARTIFACT_PATH = "slomi23-free-university-of-tbilisi-/ML_fx_LightGBM_Walmart/lightgbm-final:latest"

try:
    # 3. Fetch and download the artifact
    print(f"Fetching artifact: {ARTIFACT_PATH}")
    artifact = run.use_artifact(ARTIFACT_PATH, type="model")

    # Download to a local directory
    artifact_dir = artifact.download("lgbm_model")
    print(f"Artifact downloaded to: {artifact_dir}")

    # 4. Load the Model
    # Find the .joblib file in the downloaded directory
    model_path = None
    for root, dirs, files in os.walk(artifact_dir):
        for file in files:
            if file.endswith(".joblib"):
                model_path = os.path.join(root, file)
                break
        if model_path:
            break

    if not model_path:
        raise FileNotFoundError("No .joblib file found in the artifact.")

    model = joblib.load(model_path)
    print("XGBoost model loaded successfully from Registry!")

except Exception as e:
    print(f"Error fetching artifact: {e}")

run.finish()


Fetching artifact: slomi23-free-university-of-tbilisi-/ML_fx_LightGBM_Walmart/lightgbm-model-3:latest


wandb:   1 of 1 files downloaded.  


Artifact downloaded to: lgbm_model
XGBoost model loaded successfully from Registry!


In [27]:
import pandas as pd
import numpy as np
import joblib
cols_to_drop = ['Date', 'sales_lag_52']
X_test = test.drop(columns=cols_to_drop)
preds = model.predict(X_test)

# 4. Clip Negatives
preds = np.maximum(preds, 0)

# 5. Create Submission
submission = pd.DataFrame({
    'Id': test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].astype(str),
    'Weekly_Sales': preds.round(2)
})

submission.to_csv('lightgbm_submission.csv', index=False)
print("LightGBM submission saved.")
print(submission.head())

LightGBM submission saved.
               Id  Weekly_Sales
0  1_1_2012-11-02      29394.34
1  1_1_2012-11-09      18588.58
2  1_1_2012-11-16      17842.23
3  1_1_2012-11-23      21189.09
4  1_1_2012-11-30      16115.82


In [26]:
print(len(submission))
print(len(submission[submission['Weekly_Sales']==0]))

115064
7256


In [ ]:
print(len(submission))
print(len(submission[submission['Weekly_Sales']==0]))

115064
12377


In [ ]:
print(test['sales_lag_52'].isnull().sum())


0
